# Epithelial Subcluster Visualization with L3 Annotations v1.0

**Purpose:**
- Apply refined L3 cell type annotations to epithelial subclusters
- Generate comprehensive publication-quality visualizations
- Create marker dotplots, heatmaps, and UMAPs

**Author:** r2end  
**Date:** 2025-02-05  
**Version:** 1.0

## Configuration

In [1]:
import gc
import sys
import warnings
from pathlib import Path
from datetime import datetime
import time

import numpy as np
import pandas as pd
import scanpy as sc

warnings.filterwarnings("ignore")
sc.settings.verbosity = 1

SCRIPT_DIR = Path("/home/h2048/script/py")
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

from anndata_expression_viz_helper_20260423_v1 import (
    get_available_markers,
    prepare_output_dirs,
    resolve_expression_source,
)
from epithelial_analysis_helper_20260423_v1 import (
    apply_l3_annotations,
    build_default_l3_colors,
    build_default_l3_mapping,
    build_default_lineage_groups,
    build_default_marker_panels,
    build_misannotation_review_specs,
    make_core_marker_heatmap,
    make_lineage_dotplots,
    make_lineage_umaps,
    make_overview_umap,
    run_misannotation_review,
)

# =============================================================================
# PATHS
# =============================================================================
INPUT_H5AD = "/home/h2048/data/py/0122/epithelial_subcluster_v4_5_2_production/epithelial_with_subclusters_v4_5_2.h5ad"
MARKER_DIR = Path("/home/h2048/data/R/0129/epithelial_interpret_v2_7_FIXED")
OUTPUT_DIR = Path(f"/home/h2048/data/py/{datetime.now().strftime('%m%d')}/epithelial_viz_L3_v1_0")
output_dirs = prepare_output_dirs(OUTPUT_DIR)

# L3 annotation mapping CSV (from document 4)
L3_MAPPING_CSV = OUTPUT_DIR / "cluster_to_L3_mapping.csv"

# =============================================================================
# MARKER DEFINITIONS / SCHEMA
# =============================================================================
cluster_to_l3_mapping = build_default_l3_mapping()
L3_MARKER_PANELS = build_default_marker_panels()
L3_COLORS = build_default_l3_colors()
LINEAGE_GROUPS_UMAP = build_default_lineage_groups(kind="umap")
LINEAGE_GROUPS_DOTPLOT = build_default_lineage_groups(kind="dotplot")
MISANNOTATION_REVIEW_SPECS = build_misannotation_review_specs()

# =============================================================================
# VISUALIZATION PARAMETERS
# =============================================================================
FIGURE_DPI = 300
FIGURE_FORMAT = "pdf"
UMAP_SIZE = 3
UMAP_ALPHA = 0.6

# =============================================================================
# REPRODUCIBILITY
# =============================================================================
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

sc.settings.figdir = output_dirs["figures"]
PIPELINE_START = time.time()

print("=" * 80)
print("Epithelial Subcluster Visualization Pipeline v1.0")
print("=" * 80)
print(f"Input:   {INPUT_H5AD}")
print(f"Markers: {MARKER_DIR}")
print(f"Output:  {OUTPUT_DIR}")
print(f"Date:    {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

Epithelial Subcluster Visualization Pipeline v1.0
Input:   /home/h2048/data/py/0122/epithelial_subcluster_v4_5_2_production/epithelial_with_subclusters_v4_5_2.h5ad
Markers: /home/h2048/data/R/0129/epithelial_interpret_v2_7_FIXED
Output:  /home/h2048/data/py/0423/epithelial_viz_L3_v1_0
Date:    2026-04-23 01:32:19


## Step 1: Create L3 Mapping File

In [2]:
print("\n" + "=" * 80)
print("STEP 1: Creating L3 Annotation Mapping")
print("=" * 80)

mapping_df = pd.DataFrame(
    list(cluster_to_l3_mapping.items()),
    columns=["Cluster", "cell_type_L3_refined"],
)
mapping_df.to_csv(L3_MAPPING_CSV, index=False)

print(f"[OK] Created L3 mapping: {L3_MAPPING_CSV.name}")
print(f"  Total clusters: {len(cluster_to_l3_mapping)}")
print(f"  Unique L3 labels: {len(set(cluster_to_l3_mapping.values()))}")
print("\nL3 label distribution:")
l3_counts = pd.Series(cluster_to_l3_mapping.values()).value_counts()
for label, count in l3_counts.head(15).items():
    print(f"  {label}: {count} clusters")


STEP 1: Creating L3 Annotation Mapping
[OK] Created L3 mapping: cluster_to_L3_mapping.csv
  Total clusters: 48
  Unique L3 labels: 21

L3 label distribution:
  Squamous_Metaplasia: 7 clusters
  Goblet_Mucin: 4 clusters
  Ciliated_Mature: 4 clusters
  Basal_Cycling: 4 clusters
  AT2_Canonical: 3 clusters
  Basal_Progenitor: 3 clusters
  SMG_Serous: 3 clusters
  Ionocyte_Brush: 3 clusters
  Basal_EMT_ECM: 3 clusters
  Ciliated_Cycling_Immature: 2 clusters
  Ciliogenesis_Deuterosomal: 2 clusters
  AT1_MatrixRemodeling: 1 clusters
  AT1_Canonical: 1 clusters
  Epithelial_Cycling: 1 clusters
  AT2_Inflammatory_Repair: 1 clusters


## Step 2: Load Data and Apply L3 Annotations

In [3]:
print("\n" + "=" * 80)
print("STEP 2: Loading Data and Applying L3 Annotations")
print("=" * 80)

t0 = time.time()
adata = sc.read_h5ad(INPUT_H5AD)
print(f"[OK] Loaded in {time.time() - t0:.1f}s")

print("\nDataset summary:")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")
print(f"  Has .raw: {adata.raw is not None}")

if "subcluster" not in adata.obs.columns:
    raise ValueError("'subcluster' column not found in adata.obs!")

print("\nOriginal subcluster distribution:")
subcluster_counts = adata.obs["subcluster"].astype(str).str.strip().value_counts()
print(f"  Total subclusters: {len(subcluster_counts)}")
print(f"  Range: {subcluster_counts.min()} - {subcluster_counts.max()} cells")

print("\n[INFO] Applying L3 annotations with helper...")
adata = apply_l3_annotations(
    adata,
    cluster_to_l3_mapping,
    subcluster_col="subcluster",
    output_col="cell_type_L3",
    inplace=False,
)

mapping_summary = adata.uns["cell_type_L3_summary"]
if mapping_summary["n_unmapped"] > 0:
    print(f"[WARN] {mapping_summary['n_unmapped']} cells with unmapped subclusters:")
    for cluster in mapping_summary["unmapped_clusters"]:
        n_cells = int((adata.obs["subcluster"].astype(str).str.strip() == cluster).sum())
        print(f"  {cluster}: {n_cells} cells")
    print("[INFO] Filled unmapped cells with original stripped subcluster names")

print("\n[OK] L3 annotations applied")
l3_counts = adata.obs["cell_type_L3"].value_counts()
print(f"  Total L3 labels: {len(l3_counts)}")
print("\nL3 cell type distribution:")
for label, count in l3_counts.head(20).items():
    pct = count / adata.n_obs * 100
    print(f"  {label}: {count:,} cells ({pct:.1f}%)")
if len(l3_counts) > 20:
    print(f"  ... and {len(l3_counts) - 20} more")


STEP 2: Loading Data and Applying L3 Annotations
[OK] Loaded in 1167.3s

Dataset summary:
  Cells: 278,553
  Genes: 53,973
  Has .raw: True

Original subcluster distribution:
  Total subclusters: 48
  Range: 33 - 47038 cells

[INFO] Applying L3 annotations with helper...

[OK] L3 annotations applied
  Total L3 labels: 21

L3 cell type distribution:
  Goblet_Defense_DUOX2: 47,038 cells (16.9%)
  Basal_Progenitor: 40,407 cells (14.5%)
  Secretory_Club: 38,469 cells (13.8%)
  Ciliated_Mature: 33,260 cells (11.9%)
  Basal_Cycling: 27,041 cells (9.7%)
  Basal_EMT_ECM: 25,808 cells (9.3%)
  AT2_Canonical: 18,978 cells (6.8%)
  Squamous_Metaplasia: 11,002 cells (3.9%)
  Basal_Inflammatory: 10,883 cells (3.9%)
  Ciliated_Cycling_Immature: 5,829 cells (2.1%)
  SMG_Serous: 5,258 cells (1.9%)
  AT2_Inflammatory_Repair: 4,289 cells (1.5%)
  Goblet_Mucin: 3,304 cells (1.2%)
  Ionocyte_Brush: 2,154 cells (0.8%)
  AT1_Canonical: 1,533 cells (0.6%)
  AT1_MatrixRemodeling: 1,370 cells (0.5%)
  Secreto

## Step 3: Load Marker Data

In [ ]:
print("\n" + "=" * 80)
print("STEP 3: Loading Marker Gene Data")
print("=" * 80)

markers_all = MARKER_DIR / "all_markers.csv"
markers_top30 = MARKER_DIR / "top30_per_cluster.csv"
markers_filtered = MARKER_DIR / "top_markers_filtered.csv"

if not markers_all.exists():
    raise FileNotFoundError(f"Marker file not found: {markers_all}")

df_markers = pd.read_csv(markers_all)
print(f"[OK] Loaded markers: {markers_all.name}")
print(f"  Total genes: {len(df_markers):,}")
print(f"  Clusters: {df_markers['cluster'].nunique()}")

if 'avg_log2FC' in df_markers.columns:
    print(f"  Log2FC range: [{df_markers['avg_log2FC'].min():.2f}, {df_markers['avg_log2FC'].max():.2f}]")
if 'p_val_adj' in df_markers.columns:
    sig_markers = (df_markers['p_val_adj'] < 0.05).sum()
    print(f"  Significant markers (padj < 0.05): {sig_markers:,}")

if markers_filtered.exists():
    df_markers_filt = pd.read_csv(markers_filtered)
    print(f"\n[OK] Loaded filtered markers: {markers_filtered.name}")
    print(f"  Total genes: {len(df_markers_filt):,}")
else:
    print(f"\n[WARN] Filtered markers not found: {markers_filtered.name}")
    df_markers_filt = df_markers.copy()

## Step 4: Generate L3 Overview UMAP

In [5]:
print("\n" + "=" * 80)
print("STEP 4: Generating L3 Overview UMAP")
print("=" * 80)

overview_path = make_overview_umap(
    adata,
    output_path=sc.settings.figdir / f"01_umap_L3_overview.{FIGURE_FORMAT}",
    color_col="cell_type_L3",
    l3_colors=L3_COLORS,
    random_seed=RANDOM_SEED,
    figure_dpi=FIGURE_DPI,
    umap_size=UMAP_SIZE,
    umap_alpha=UMAP_ALPHA,
)
print(f"[OK] Saved: {overview_path.name}")

print("\n[INFO] Generating lineage-specific UMAPs with helper...")
lineage_path = make_lineage_umaps(
    adata,
    output_path=sc.settings.figdir / f"02_umap_L3_by_lineage.{FIGURE_FORMAT}",
    lineage_groups=LINEAGE_GROUPS_UMAP,
    color_col="cell_type_L3",
    random_seed=RANDOM_SEED,
    figure_dpi=FIGURE_DPI,
    umap_size=UMAP_SIZE,
    umap_alpha=UMAP_ALPHA,
)
print(f"[OK] Saved: {lineage_path.name}")


STEP 4: Generating L3 Overview UMAP
[OK] Saved: 01_umap_L3_overview.pdf

[INFO] Generating lineage-specific UMAPs with helper...
[OK] Saved: 02_umap_L3_by_lineage.pdf


In [4]:
print("\n" + "=" * 80)
print("STEP 5B: Misannotation Review Dotplot")
print("=" * 80)

review_results = run_misannotation_review(
    adata,
    output_dir=OUTPUT_DIR,
    specs=MISANNOTATION_REVIEW_SPECS,
    cluster_col="subcluster",
    figure_format=FIGURE_FORMAT,
    figure_dpi=FIGURE_DPI,
)

print("[INFO] Review clusters:")
for cluster, count in review_results["review_counts"].items():
    print(f"  {cluster}: {count:,} cells")

print(f"[OK] Saved: {review_results['plot_path'].name}")
print(f"[OK] Review tables: {output_dirs['tables']}")

summary_cols = ["cluster", "side", "label", "available_genes", "missing_genes"]
print("\n[INFO] Review panel summary:")
print(review_results["section_df"][summary_cols].to_string(index=False))

_ = gc.collect()


STEP 5B: Misannotation Review Dotplot
[INFO] Review clusters:
  AT1 _2: 37 cells
  Ciliated_5: 144 cells
  SMG_Mucous_1: 57 cells
  SMG_Basal_1: 66 cells
  Secretory_Goblet_2: 29,239 cells
  Secretory_Goblet_3: 1,031 cells
  Secretory_Goblet_4: 98 cells
[OK] Saved: 03b_dotplot_misannotation_review.pdf
[OK] Review tables: /home/h2048/data/py/0423/epithelial_viz_L3_v1_0/tables

[INFO] Review panel summary:
           cluster    side                     label                        available_genes missing_genes
            AT1 _2   wrong             AT1_Canonical               AGER;HOPX;CAV1;AQP4;EMP2              
            AT1 _2 correct             AT2_Canonical  SFTPC;SFTPA1;SFTPA2;SFTPB;ABCA3;NAPSA              
        Ciliated_5   wrong           Ciliated_Mature          FOXJ1;TPPP3;DNAH5;DNAH9;RSPH1              
        Ciliated_5 correct              Goblet_Mucin    SPDEF;FOXA3;MUC5AC;MUC5B;FCGBP;TFF3              
      SMG_Mucous_1   wrong              Goblet_Mucin    SPDEF

## Step 5: Generate Marker Dotplots

In [6]:
print("\n" + "=" * 80)
print("STEP 5: Generating Marker Dotplots")
print("=" * 80)

source = resolve_expression_source(adata)
available_markers, missing_markers = get_available_markers(L3_MARKER_PANELS, source["gene_universe"])

print("\n[INFO] Marker availability:")
print(f"  Expression source: {source['source_name']}")
print(f"  L3 categories with markers: {len(available_markers)}/{len(L3_MARKER_PANELS)}")
print(f"  Total available markers: {sum(len(v) for v in available_markers.values())}")
print(f"  Missing markers: {len(missing_markers)}")

if 0 < len(missing_markers) <= 20:
    print(f"  Missing: {', '.join(sorted(missing_markers))}")
elif len(missing_markers) > 20:
    print(f"  Missing (first 20): {', '.join(sorted(missing_markers)[:20])}")
    print(f"  ... and {len(missing_markers) - 20} more")

print("\n[INFO] Generating lineage-specific dotplots with helper...")
dotplot_paths = make_lineage_dotplots(
    adata,
    output_dir=OUTPUT_DIR,
    marker_panels=L3_MARKER_PANELS,
    lineage_groups=LINEAGE_GROUPS_DOTPLOT,
    color_col="cell_type_L3",
    figure_format=FIGURE_FORMAT,
    figure_dpi=FIGURE_DPI,
)

for lineage, path in dotplot_paths.items():
    print(f"  [OK] {lineage}: {path.name}")

missing_lineages = sorted(set(LINEAGE_GROUPS_DOTPLOT) - set(dotplot_paths))
if missing_lineages:
    print(f"[WARN] No dotplot generated for: {', '.join(missing_lineages)}")

_ = gc.collect()


STEP 5: Generating Marker Dotplots

[INFO] Marker availability:
  Expression source: .raw
  L3 categories with markers: 21/21
  Total available markers: 135
  Missing markers: 0

[INFO] Generating lineage-specific dotplots with helper...
  [OK] Alveolar: 03_dotplot_Alveolar.pdf
  [OK] Basal: 03_dotplot_Basal.pdf
  [OK] Ciliated: 03_dotplot_Ciliated.pdf
  [OK] Secretory_SMG: 03_dotplot_Secretory_SMG.pdf
  [OK] Special: 03_dotplot_Special.pdf


## Step 6: Generate Comprehensive Marker Heatmap

In [7]:
print("\n" + "=" * 80)
print("STEP 6: Generating Marker Expression Heatmap")
print("=" * 80)

all_core_markers = sorted({gene for markers in available_markers.values() for gene in markers})
print(f"[INFO] Total core markers: {len(all_core_markers)}")

heatmap_path = make_core_marker_heatmap(
    adata,
    output_path=sc.settings.figdir / f"04_heatmap_core_markers.{FIGURE_FORMAT}",
    marker_panels=L3_MARKER_PANELS,
    color_col="cell_type_L3",
    figure_dpi=FIGURE_DPI,
)

if heatmap_path is None:
    print("[WARN] No expression data available for heatmap")
else:
    print(f"[OK] Saved: {heatmap_path.name}")

_ = gc.collect()


STEP 6: Generating Marker Expression Heatmap
[INFO] Total core markers: 104
[OK] Saved: 04_heatmap_core_markers.pdf


## Step 7: Generate Marker Expression UMAPs